# What can one battery ageing record support? — Google Colab reproduction
**Results in Engineering research article** (co-submitted with the TRACE method article in *MethodsX*).
Running all cells regenerates every number, table and figure of the article (Figs. 1–8, Tables 4–13):
index-only ceiling, NASA engineering reading, feature atlas and 3-D landscape, cross-cell transfer with robustness
controls, identifiability vs transfer, and SHAP explainability / explanation reliability.

* **GMP LiFePO₄ dataset** (Mendeley Data, doi:10.17632/k675pxrd83.2): upload `GMP_LFP_40Ah_3000Cycle.csv` in Step 1, or skip to regenerate it from the published equations.
* **NASA PCoE cells B0005/6/7/18**: downloaded automatically.
* **374 public cells (XJTU, TJU, HUST, MIT)**: cloned automatically from github.com/wang-fujin/PINN4SOH (commit cc3cc50).
* `QUICK = True` gives a short smoke test; `False` reproduces the article (about 1.5–2 h on a free Colab CPU).

In [ ]:
QUICK = False   # True = fast smoke test; False = exact article numbers
import os; os.environ['TRACE_QUICK'] = '1' if QUICK else '0'; os.environ['PYTHONPATH'] = '.'
!pip -q install pyreadr==0.5.9 pymannkendall==1.4.3 shap statsmodels scikit-learn joblib
!mkdir -p data outputs figures

## Step 1 – (optional) upload the published GMP CSV

In [ ]:
try:
    from google.colab import files
    up = files.upload()
    for name in up:
        os.replace(name, 'data/GMP.csv'); print('saved data/GMP.csv')
except Exception as e:
    print('No upload (', e, ') - the dataset will be regenerated from the published equations.')

## Step 2 – the TRACE audit module

In [ ]:
%%writefile trace_audit.py
"""
TRACE: Trajectory-Referenced Audit of Cycle-level Evidence
A pre-modelling information audit for cycle-indexed battery prognostics datasets.

Stages
  S1  Provenance / identity screen  -> determinism index eta, exact identities
  S2  Index-surrogate ceiling        -> skill attainable from the cycle index alone
  S3  Association tests (time-shift) -> marginal and index-conditional, Holm-corrected
  S4  Sensitivity calibration        -> injection power curves, minimum detectable effect

Dependencies: numpy, scipy, pandas (scikit-learn optional, not required).
"""
from __future__ import annotations
import numpy as np
import pandas as pd
from scipy import stats

from scipy.optimize import curve_fit

# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------

def _z(a):
    a = np.asarray(a, float)
    s = a.std()
    return (a - a.mean()) / s if s > 0 else a * 0.0


_PROJ_CACHE = {}


def _bspline_basis(k, n_knots):
    from scipy.interpolate import BSpline
    deg = 3
    lo, hi = k.min(), k.max()
    inner = np.quantile(k, np.linspace(0, 1, n_knots + 2)[1:-1])
    t = np.r_[[lo] * (deg + 1), inner, [hi] * (deg + 1)]
    return BSpline.design_matrix(np.clip(k, lo, hi), t, deg).toarray()


def _crossfit_operator(k, n_knots):
    """Linear operator M such that M @ x is the two-fold interleaved cross-fitted
    cubic regression-spline trend of x on k (cached per index vector)."""
    key = (len(k), float(k[0]), float(k[-1]), n_knots, float(np.sum(k)))
    if key in _PROJ_CACHE:
        return _PROJ_CACHE[key]
    B = _bspline_basis(k, n_knots)
    n = len(k); idx = np.arange(n); M = np.zeros((n, n))
    for part in (0, 1):
        tr = idx % 2 != part; te = ~tr
        P = np.linalg.pinv(B[tr])            # least-squares coefficients map
        M[np.ix_(te, tr)] = B[te] @ P
    _PROJ_CACHE[key] = M
    return M


def crossfit_trend(k, x, n_knots=None):
    """Two-fold interleaved cross-fitted cubic regression spline of x on index k.
    Points at even positions are predicted from a spline fitted to odd positions and
    vice versa, so the trend cannot absorb observation noise.
    Default: min(12, n//15) interior quantile knots. Returns (trend, residual)."""
    k = np.asarray(k, float); x = np.asarray(x, float)
    if n_knots is None:
        n_knots = int(max(3, min(12, len(k) // 15)))
    M = _crossfit_operator(k, n_knots)
    fit = M @ x
    return fit, x - fit


# ----------------------------------------------------------------------------
# S1  provenance / identity screen
# ----------------------------------------------------------------------------

def determinism_index(k, x):
    """eta = RMS(cross-fitted residual) / SD(x).
    eta ~ 1: no smooth dependence on the index (stationary column);
    eta << 1 (< 1e-3): column is a deterministic smooth function of the index."""
    x = np.asarray(x, float)
    sd = x.std()
    if sd == 0:
        return np.nan
    _, r = crossfit_trend(k, x)
    return float(np.sqrt(np.mean(r ** 2)) / sd)


def identity_screen(df: pd.DataFrame, index_col: str, tol=1e-9):
    """Flag constant columns, columns affine in another column (|r| >= 1-tol),
    and columns strictly monotone in the index."""
    out = []
    cols = [c for c in df.columns]
    k = df[index_col].values
    for c in cols:
        x = df[c].values.astype(float)
        rec = dict(column=c)
        rec['constant'] = bool(np.nanstd(x) == 0)
        if rec['constant']:
            rec.update(affine_to='', strictly_monotone_in_index=False,
                       spearman_index=np.nan, eta=np.nan)
            out.append(rec); continue
        partners = []
        for c2 in cols:
            if c2 == c or np.std(df[c2].values) == 0:
                continue
            r = np.corrcoef(x, df[c2].values.astype(float))[0, 1]
            if abs(r) >= 1 - tol:
                partners.append(c2)
        rec['affine_to'] = ';'.join(partners)
        dx = np.diff(x[np.argsort(k)])
        rec['strictly_monotone_in_index'] = bool(np.all(dx > 0) or np.all(dx < 0))
        rec['spearman_index'] = float(stats.spearmanr(k, x)[0])
        rec['eta'] = determinism_index(k, x) if c != index_col else 0.0
        out.append(rec)
    return pd.DataFrame(out)


def trend_r2(k, x):
    """Cross-fitted trend R^2 = 1 - eta^2 (can be slightly negative under no trend)."""
    e = determinism_index(k, x)
    return np.nan if np.isnan(e) else 1 - e ** 2


def _lb_q(R, lags):
    """Ljung-Box Q for each column of R."""
    R = R - R.mean(0); n = R.shape[0]; d = np.sum(R ** 2, 0)
    return n * (n + 2) * sum(np.sum(R[h:] * R[:-h], 0) ** 2 / d ** 2 / (n - h) for h in range(1, lags + 1))


def lb_gate(k, x, lags=10, B=999, seed=0):
    """Exchangeability gate: Ljung-Box statistic of the spline-detrended residual, with its null
    distribution obtained by passing i.i.d. N(0,1) series through the same detrending projection.
    The asymptotic chi-square reference ignores the degrees of freedom removed by the spline and
    rejects white noise far too often on short records (34% at n = 168); this calibration restores
    the nominal size under i.i.d. errors. Returns the Monte Carlo p-value."""
    k = np.asarray(k, float); x = np.asarray(x, float)
    Bf = _bspline_basis(k, int(max(3, min(12, len(k) // 15))))
    Q, _ = np.linalg.qr(Bf)
    proj = lambda V: V - Q @ (Q.T @ V)
    q0 = _lb_q(proj(x[:, None]), lags)[0]
    Z = np.random.default_rng(seed + 7919).normal(size=(len(k), B))
    qn = _lb_q(proj(Z), lags)
    return float((1 + np.sum(qn >= q0)) / (B + 1))


def trend_test(k, x, B=999, seed=0, lb_lags=10):
    """S1b: permutation test of H0 'no dependence of x on the cycle index'.
    Statistic: cross-fitted trend R^2. Null: random permutations of x (valid when
    the observations are exchangeable). Exchangeability is checked by a Ljung-Box
    test on the detrended residual; if it fails, the result is descriptive only."""
    k = np.asarray(k, float); x = np.asarray(x, float)
    M = _crossfit_operator(k, int(max(3, min(12, len(k) // 15))))
    def stat(v):
        r = v - M @ v
        return 1 - np.mean(r ** 2) / np.var(v)
    s0 = stat(x)
    rng = np.random.default_rng(seed)
    P = np.stack([rng.permutation(x) for _ in range(B)], 1)
    R = P - M @ P
    null = 1 - np.mean(R ** 2, 0) / np.var(P, 0)
    p = (1 + np.sum(null >= s0)) / (B + 1)
    # exchangeability gate on the full-sample detrended residual (interleaved cross-fitting itself
    # induces lag-1 structure), calibrated by Monte Carlo for the spline projection (lb_gate)
    lb_p = lb_gate(k, x, lags=lb_lags, seed=seed)
    return dict(R2_trend=float(s0), p_trend=float(p), null_q95=float(np.quantile(null, .95)),
                null_q99=float(np.quantile(null, .99)), LB_p=lb_p, exchangeable=lb_p > 0.05)


# ----------------------------------------------------------------------------
# S2  index-surrogate ceiling
# ----------------------------------------------------------------------------

def _powerlaw(k, a, b, c):
    return a - b * np.power(k, c)


def index_ceiling(k, y, train_frac=0.7):
    """Chronological split. Fits y = a - b*k^c on the first train_frac of cycles
    using the index only and extrapolates to the rest. Returns test R2, RMSE.
    High R2 means the target is (near-)fully determined by the index: any
    feature-based model must be judged against this ceiling."""
    k = np.asarray(k, float); y = np.asarray(y, float)
    n = len(k); m = int(round(train_frac * n))
    kt, yt = k[:m], y[:m]
    try:
        p0 = (yt[0], (yt[0] - yt[-1]) / max(kt[-1] ** 1.0, 1), 1.0)
        p, _ = curve_fit(_powerlaw, kt, yt, p0=p0, maxfev=20000,
                         bounds=([-np.inf, -np.inf, 0.2], [np.inf, np.inf, 4.0]))
        yhat = _powerlaw(k[m:], *p)
    except Exception:
        coef = np.polyfit(kt, yt, 2)
        yhat = np.polyval(coef, k[m:])
    ytest = y[m:]
    ss_res = np.sum((ytest - yhat) ** 2)
    ss_tot = np.sum((ytest - ytest.mean()) ** 2)
    return dict(R2_index=float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
                RMSE_index=float(np.sqrt(np.mean((ytest - yhat) ** 2))))


# ----------------------------------------------------------------------------
# S3  association tests: conservative truncated time-shift test
# ----------------------------------------------------------------------------

def _absspearman(a, b):
    return abs(stats.spearmanr(a, b)[0])


def _shift_corr(x, y, r):
    """|Pearson r| between x[r+d : n-r+d] and y[r : n-r] for every d in [-r, r],
    computed in O(n log n) with FFT cross-correlation and cumulative sums."""
    from scipy.signal import correlate
    n = len(x); L = n - 2 * r
    yw = y[r:n - r]
    yc = yw - yw.mean(); syy = np.sum(yc ** 2)
    sxy = correlate(x, yc, mode='valid', method='fft')        # length 2r+1
    cs = np.concatenate([[0.0], np.cumsum(x)])
    cs2 = np.concatenate([[0.0], np.cumsum(x * x)])
    sx = cs[L:] - cs[:-L]; sxx = cs2[L:] - cs2[:-L]
    vx = sxx - sx ** 2 / L
    with np.errstate(invalid='ignore', divide='ignore'):
        S = np.abs(sxy) / np.sqrt(vx * syy)
    return np.nan_to_num(S)


TTS_DIFF_DEFAULT = True   # TRACE v2: level + rate representation of the target residual


def tts_test(x, y, r=None, rank=True, lags=0, diff=None):
    """Truncated time-shift test (Harris 2020; Yuan & Shou 2024), conservative.
    x is the series assumed stationary; y is held fixed. For each shift d in [-r, r] the
    statistic is computed between x[R+d : n-R+d] and the fixed window y[R : n-R], R = r + lags.
    Basic statistic: |rank correlation| at lag 0 (lags=0, diff=False; TRACE v1).
    TRACE v2 statistic: the maximum |rank correlation| over lags l in [-lags, lags] and over two
    representations of the target, its level residual and its first difference (rate). The same
    statistic is computed at every shift, so the test remains valid for any stationary x.
    p = #{d : S_d >= S_0} / (2r+1); minimum attainable p = 1/(2r+1). Returns (S0, p)."""
    if diff is None:
        diff = TTS_DIFF_DEFAULT
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    if r is None:
        r = n // 4
    R = r + lags
    targets = [y] + ([np.r_[0.0, np.diff(y)]] if diff else [])
    if rank:
        x = stats.rankdata(x); targets = [stats.rankdata(t) for t in targets]
    best = None
    for t in targets:
        S = _shift_corr(x, t, R)                     # shifts -R..R
        if lags:
            S = np.max(np.stack([S[lags + l: lags + l + 2 * r + 1] for l in range(-lags, lags + 1)]), 0)
        best = S if best is None else np.maximum(best, S)
    S0 = best[r]
    if lags:
        # exclusion zone: shifts with |d| <= 2*lags share alignments with d = 0 and are left out of the null
        d = np.arange(-r, r + 1); null = best[np.abs(d) > 2 * lags]
        p = (1 + np.sum(null >= S0 - 1e-12)) / (len(null) + 1)
    else:
        p = np.sum(best >= S0 - 1e-12) / (2 * r + 1)
    return float(S0), float(p)


def holm(p):
    p = np.asarray(p, float); m = len(p)
    o = np.argsort(p); adj = np.empty(m); run = 0.0
    for i, j in enumerate(o):
        run = max(run, min(1.0, (m - i) * p[j]))
        adj[j] = run
    return adj


def association_tests(k, y, X: pd.DataFrame, r=None, conditional=True):
    """Marginal (x vs y) and index-conditional (resid_x vs resid_y) TTS tests."""
    rows = []
    _, ry = crossfit_trend(k, y)
    ry_ok = np.std(ry) > 1e-6 * np.std(y)
    for c in X.columns:
        x = X[c].values.astype(float)
        S0, p = tts_test(x, y, r)
        rec = dict(feature=c, S_marg=S0, p_marg=p)
        if conditional and ry_ok:
            _, rx = crossfit_trend(k, x)
            S0c, pc = tts_test(rx, ry, r)
            rec.update(S_cond=S0c, p_cond=pc)
        else:
            rec.update(S_cond=np.nan, p_cond=np.nan)
        rows.append(rec)
    out = pd.DataFrame(rows)
    out['p_marg_holm'] = holm(out.p_marg.values)
    if out.p_cond.notna().all():
        out['p_cond_holm'] = holm(out.p_cond.values)
    else:
        out['p_cond_holm'] = np.nan
    out.attrs['conditional_testable'] = bool(ry_ok)
    return out


# ----------------------------------------------------------------------------
# S4  sensitivity calibration by signal injection (self-surrogate)
# ----------------------------------------------------------------------------

def inject(xnull, target, rho):
    """Return x_inj whose population correlation with target is rho when xnull
    is independent of target; preserves the mean/SD of xnull."""
    zx, zt = _z(xnull), _z(target)
    zi = np.sqrt(1 - rho ** 2) * zx + rho * zt
    return xnull.mean() + xnull.std() * zi


def self_surrogate_power(x, y, rhos, n_rep=200, alpha=0.05, r=None, seed=0,
                         target=None, n_tests=1):
    """Power of the (Holm-adjusted, n_tests-family) TTS test to detect an injected
    coupling of strength rho. Null realisations are random circular shifts of the
    observed x (self-surrogates), which preserve its autocorrelation."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float); y = np.asarray(y, float)
    t = y if target is None else target
    n = len(x)
    a_eff = alpha / n_tests  # Bonferroni bound on the Holm step for the worst rank
    res = []
    for rho in rhos:
        hits = 0
        for _ in range(n_rep):
            xs = np.roll(x, rng.integers(n // 10, n - n // 10))
            xi = inject(xs, t, rho)
            _, p = tts_test(xi, y, r)
            hits += p <= a_eff
        res.append(hits / n_rep)
    return np.array(res)


def mde(rhos, power, target=0.8):
    """Interpolated smallest coupling with power >= target (MDE80)."""
    rhos = np.asarray(rhos); power = np.asarray(power)
    above = np.where(power >= target)[0]
    if len(above) == 0:
        return np.nan
    i = above[0]
    if i == 0:
        return float(rhos[0])
    # linear interpolation between grid points
    r0, r1, p0, p1 = rhos[i - 1], rhos[i], power[i - 1], power[i]
    return float(r0 + (target - p0) * (r1 - r0) / (p1 - p0)) if p1 > p0 else float(r1)


# ----------------------------------------------------------------------------
# verdicts
# ----------------------------------------------------------------------------

def verdict(row, mde_val, rho_rel=0.1, alpha=0.05, derived=False):
    if derived:
        return 'D'  # derived / deterministic: exclude as feature
    if row['p_marg_holm'] <= alpha or (not np.isnan(row.get('p_cond_holm', np.nan))
                                       and row['p_cond_holm'] <= alpha):
        return 'I'  # informative
    if not np.isnan(mde_val) and mde_val <= rho_rel:
        return 'N'  # null with adequate power
    return 'U'      # underpowered / inconclusive


# ----------------------------------------------------------------------------
# full audit card
# ----------------------------------------------------------------------------

def proxy_check(k, y, x):
    """Residual correlation between feature and target after removing the index
    trend. |r_res| >= 0.98 means the feature reproduces the target's own
    cycle-to-cycle fluctuations: a likely co-computed proxy (verify provenance)."""
    _, ry = crossfit_trend(k, y); _, rx = crossfit_trend(k, x)
    if np.std(ry) < 1e-6 * np.std(y):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def restatement_index(x, y):
    """zeta = RMS(y - a - b x) / SD(y): how closely a linear map of the feature restates the target on
    the raw scale. zeta < 0.01 means the feature reproduces the target to within 1% of its spread."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    A = np.column_stack([np.ones_like(x), x]); res = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
    return float(np.sqrt(np.mean(res ** 2)) / np.std(y))


def run_audit(df, index_col, target_col, feature_cols, r=None, B=999,
              eta_det=1e-3, proxy_thr=0.98, zeta_thr=0.01, alpha=0.05, seed=0, version=2,
              mde=False, mde_rhos=(0, .1, .2, .3, .4, .5, .6, .7, .8, .9), mde_reps=200):
    """TRACE audit card (stages S1-S4).
    S1 identity / determinism (eta) / proxy (|r_res|, zeta) screen; S2 index-only ceiling (attrs['ceiling']);
    S3 gated permutation trend test and truncated time-shift beyond-index test, Holm within the tested family;
    S4 (mde=True) self-surrogate MDE80 of the beyond-index test for every T and N feature (column 'MDE80').
    The truncation radius r is fixed before classification, so the resolution rule uses m = number of candidate
    features (an upper bound on the tested family); Holm then uses the tested family.
    version=2 (default): level + rate beyond-index statistic and restatement index zeta. version=1: original TRACE."""
    lags, use_diff = (0, True) if version >= 2 else (0, False)
    k = df[index_col].values.astype(float); y = df[target_col].values.astype(float)
    n = len(k)
    m_tests = len(feature_cols)
    if r is None:
        # resolution rule: the smallest Holm-adjusted p, m/(2r+1), must not exceed alpha
        r_floor = int(np.ceil((m_tests / alpha - 1) / 2))
        r = int(min(max(n // 4, r_floor), np.floor(0.4 * n) - lags))
    eta_y = determinism_index(k, y)
    ceil = index_ceiling(k, y)
    rows = []
    for c in feature_cols:
        x = df[c].values.astype(float)
        rec = dict(feature=c)
        if np.std(x) == 0:
            rec.update(cls='D', reason='constant'); rows.append(rec); continue
        rx_aff = abs(np.corrcoef(x, k)[0, 1]) >= 1 - 1e-9 or abs(np.corrcoef(x, y)[0, 1]) >= 1 - 1e-9
        eta = determinism_index(k, x)
        tt = trend_test(k, x, B=B, seed=seed)
        rp = proxy_check(k, y, x)
        zt = restatement_index(x, y) if version >= 2 else np.nan
        rec.update(eta=eta, R2_trend=tt['R2_trend'], p_trend=tt['p_trend'],
                   exch=tt['exchangeable'], r_resid=rp, zeta=zt)
        if eta_y < eta_det:
            rec.update(S_cond=np.nan, p_cond=np.nan)
        else:
            _, ry = crossfit_trend(k, y); _, rx = crossfit_trend(k, x)
            S, p = tts_test(rx, ry, r, lags=lags, diff=use_diff)
            rec.update(S_cond=S, p_cond=p)
        if rx_aff:
            rec['reason'] = 'affine in index or target'
        elif eta < eta_det:
            rec['reason'] = 'deterministic in index'
        elif not np.isnan(rp) and abs(rp) >= proxy_thr:
            rec['reason'] = 'target proxy (residual |r|>=%.2f)' % proxy_thr
        elif not np.isnan(zt) and zt < zeta_thr:
            rec['reason'] = 'target proxy (restatement zeta<%.3f)' % zeta_thr
        else:
            rec['reason'] = ''
        rows.append(rec)
    out = pd.DataFrame(rows)
    tested = out['reason'].fillna('') == ''
    out['p_cond_holm'] = np.nan
    if tested.any() and out.loc[tested, 'p_cond'].notna().all():
        out.loc[tested, 'p_cond_holm'] = holm(out.loc[tested, 'p_cond'].values)
    out['p_trend_holm'] = np.nan
    ex = tested & out['exch'].fillna(False).astype(bool)
    if ex.any():
        out.loc[ex, 'p_trend_holm'] = holm(out.loc[ex, 'p_trend'].values)
    cls = []
    for _, rw in out.iterrows():
        if rw.get('reason', ''):
            cls.append('D' if 'proxy' not in rw['reason'] else 'P'); continue
        if not np.isnan(rw['p_cond_holm']) and rw['p_cond_holm'] <= alpha:
            cls.append('I'); continue
        trending = (rw['exch'] and rw['p_trend_holm'] <= alpha) or (not rw['exch'] and rw['R2_trend'] > 0.1)
        if trending:
            cls.append('T'); continue
        cls.append('N' if eta_y >= eta_det else 'N*')
    out['class'] = cls
    m_eff = int(tested.sum())
    out['MDE80'] = np.nan
    if mde and eta_y >= eta_det and m_eff > 0:
        _, ry = crossfit_trend(k, y)
        for i in out.index[out['class'].isin(['T', 'N'])]:
            _, rx = crossfit_trend(k, df[out.at[i, 'feature']].values.astype(float))
            pw = self_surrogate_power(rx, ry, mde_rhos, n_rep=mde_reps, alpha=alpha, r=r, seed=seed, n_tests=m_eff)
            out.at[i, 'MDE80'] = mde_fn(mde_rhos, pw)
    out.attrs.update(eta_target=eta_y, ceiling=ceil, r=r, conditional_testable=eta_y >= eta_det,
                     m_tested=m_eff, min_holm_p=m_eff / (2 * r + 1 - (4 * lags if lags else 0)))
    return out


mde_fn = mde   # alias used inside run_audit, whose argument 'mde' is a switch


## Step 3 – data preparation: GMP record, NASA discharge features (methods Step 1)

In [ ]:
%%writefile 00_data.py
"""Step 0 - data. GMP: use data/GMP.csv if present (download from Mendeley, doi:10.17632/k675pxrd83.2),
otherwise regenerate it from the published generating equations (Data in Brief 69 (2026) 113252, Eqs. 1-8, seed 42)
and, if the CSV is present, verify that the equations reproduce it. NASA: download data_NASA.rda and build cycle features."""
import os, urllib.request, numpy as np, pandas as pd
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True); os.makedirs('figures', exist_ok=True)

def generate_gmp(seed=42):
    np.random.seed(seed); N = 3000; n = np.arange(1, N + 1)
    cap = 40.0 - 12.0 * (n / N) ** 1.35
    V = np.random.normal(25.60, 0.12, N); I = np.random.uniform(-40.0, 40.0, N)
    Tt = np.random.normal(28.0, 1.8, N); C = np.random.normal(3.200, 0.008, (N, 8))
    d = pd.DataFrame({'PackID': 1, 'CycleNumber': n, 'Capacity(Ah)': cap, 'SOH(%)': cap / 40 * 100,
                      'RUL(cycles)': N - n, 'Voltage(V)': V, 'Current(A)': I, 'DoD(%)': n / N * 100, 'Temperature(C)': Tt})
    for i in range(8): d[f'Cell{i+1}_V'] = C[:, i]
    return d

gen = generate_gmp()
if os.path.exists('data/GMP.csv'):
    pub = pd.read_csv('data/GMP.csv')
    diff = max(float(np.max(np.abs(pub[c].values - gen[c].values))) for c in gen.columns)
    print(f'GMP: published CSV found; max |published - regenerated| = {diff:.2e}')
else:
    gen.to_csv('data/GMP.csv', index=False)
    print('GMP: published CSV not found -> regenerated from the published equations (identical to Mendeley V2 up to CSV rounding)')

if not os.path.exists('data/nasa_cycle_features.csv'):
    url = 'https://raw.githubusercontent.com/cjrincon/cnnlstm/master/data/data_NASA.rda'
    if not os.path.exists('data/data_NASA.rda'):
        urllib.request.urlretrieve(url, 'data/data_NASA.rda')
    import pyreadr
    d = pyreadr.read_r('data/data_NASA.rda')['data_NASA']; rows = []
    for (b, c), g in d.groupby(['battery', 'cycle']):
        g = g.sort_values('time'); t = g.time.values; v = g.voltage_measured.values
        T = g.temperature_measured.values; I = g.current_measured.values
        below = np.where(v <= 2.7)[0]; icut = below[0] if len(below) else len(v) - 1; seg = slice(1, icut + 1)
        rows.append(dict(battery=int(b), cycle=int(c), capacity=float(g.capacity.iloc[0]),
                         ambient_T=float(g.ambient_temperature.iloc[0]), t_discharge=float(t[icut]),
                         V_mean=float(np.mean(v[seg])), V_600s=float(np.interp(600, t, v)),
                         T_max=float(np.max(T[:icut + 1])), t_Tmax=float(t[np.argmax(T[:icut + 1])]),
                         T_start=float(T[0]), I_mean=float(np.mean(I[seg]))))
    pd.DataFrame(rows).sort_values(['battery', 'cycle']).to_csv('data/nasa_cycle_features.csv', index=False)
f = pd.read_csv('data/nasa_cycle_features.csv'); print('NASA cycles per cell:', f.groupby('battery').size().to_dict())


In [ ]:
!PYTHONPATH=. python 00_data.py

## Step 4 – audit of the GMP record and the NASA cells (methods Step 2)

In [ ]:
%%writefile 01_audits.py
import numpy as np, pandas as pd, sys, json
sys.path.insert(0,'.'); import trace_audit as T
from scipy import stats
pd.set_option('display.width',250); pd.set_option('display.max_columns',30)
g=pd.read_csv('data/GMP.csv')
cells=[f'Cell{i}_V' for i in range(1,9)]
g['Imbalance(V)']=g[cells].max(1)-g[cells].min(1)
feats=['SOH(%)','RUL(cycles)','DoD(%)','PackID','Voltage(V)','Current(A)','Temperature(C)']+cells+['Imbalance(V)']
a=T.run_audit(g,'CycleNumber','Capacity(Ah)',feats,B=999)
print(a.round(4).to_string()); print(a.attrs)
a.to_csv('outputs/out_audit_gmp.csv',index=False); json.dump({k:(v if not isinstance(v,(np.floating,np.bool_)) else float(v)) for k,v in a.attrs.items()},open('outputs/out_audit_gmp_attrs.json','w'),default=float)
f=pd.read_csv('data/nasa_cycle_features.csv')
nf=['ambient_T','I_mean','t_discharge','t_Tmax','V_mean','V_600s','T_max','T_start']
allc=[]
for b,h in f.groupby('battery'):
    h=h.reset_index(drop=True)
    a=T.run_audit(h,'cycle','capacity',nf,B=999)
    a.insert(0,'battery',f'B{b:04d}'); a['eta_target']=a.attrs['eta_target']; a['R2_index']=a.attrs['ceiling']['R2_index']; a['r']=a.attrs['r']
    allc.append(a); print(a.round(4).to_string())
A=pd.concat(allc); A.to_csv('outputs/out_audit_nasa.csv',index=False)
# (no pooling of p-values across cells: the four records are audited and reported separately)


In [ ]:
!PYTHONPATH=. python 01_audits.py

## Step 5 – audit of the 374 public cells (methods Step 2)

In [ ]:
%%writefile 11_external.py
"""Step 11 - V8: external validation on 387 measured cells from four public datasets (XJTU, TJU, HUST, MIT;
processed release github.com/wang-fujin/PINN4SOH, commit cc3cc50), each audited as its own single trajectory.
(a) Audit every cell (16 charge features, target = measured discharge capacity); store statistics and classes.
(b) Consequence for generalizability: within each dataset, leave-one-cell-out GBR with
    index only / index + features that TRACE classes I in the majority of training cells / index + T-majority
    features / index + all non-leaking features. Shows that T features, which TRACE cannot credit on one
    trajectory, may still transfer across cells (the identifiability boundary), with cell-level bootstrap CIs."""
import os, sys, glob, subprocess, warnings, numpy as np, pandas as pd
from joblib import Parallel, delayed
warnings.filterwarnings('ignore'); sys.path.insert(0, '.'); import trace_audit as T
QUICK = os.environ.get('TRACE_QUICK') == '1'
REPO, COMMIT = 'https://github.com/wang-fujin/PINN4SOH.git', 'cc3cc5053caf38f04e0665f7f88cb109144d035e'
F = ['voltage mean', 'voltage std', 'voltage kurtosis', 'voltage skewness', 'CC Q', 'CC charge time', 'voltage slope',
     'voltage entropy', 'current mean', 'current std', 'current kurtosis', 'current skewness', 'CV Q', 'CV charge time',
     'current slope', 'current entropy']


def load():
    tgt = 'data/PINN4SOH'
    if not os.path.isdir(tgt):
        subprocess.run(['git', 'clone', '-q', REPO, tgt], check=True); subprocess.run(['git', '-C', tgt, 'checkout', '-q', COMMIT], check=True)
    B = tgt + '/data/'; fs = glob.glob(B + 'XJTU data/*.csv') + glob.glob(B + 'HUST data/*.csv') + glob.glob(B + 'MIT data/*/*.csv') + glob.glob(B + 'TJU data/*/*.csv')
    out = []
    for f in sorted(fs):
        d = pd.read_csv(f).replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
        if len(d) < 100: continue
        ds = 'XJTU' if 'XJTU' in f else 'HUST' if 'HUST' in f else 'MIT' if 'MIT' in f else 'TJU'
        d = d.assign(cell=f.split('data/', 2)[-1], dataset=ds, k=np.arange(1, len(d) + 1))
        out.append(d)
    return pd.concat(out, ignore_index=True)


def audit(cell, g):
    card = T.run_audit(g[['k', 'capacity'] + F].reset_index(drop=True), 'k', 'capacity', F, B=199, seed=0)
    card['cell'] = cell; card['dataset'] = g.dataset.iloc[0]; card['n'] = len(g)
    card['eta_target'] = card.attrs['eta_target']; card['r'] = card.attrs['r']; card['p_floor_holm'] = card.attrs['min_holm_p']
    card['R2_index'] = card.attrs['ceiling']['R2_index']
    return card


if __name__ == '__main__':
    D = load(); cells = list(D.groupby('cell'))
    if QUICK: cells = cells[::8]
    print('cells', len(cells), 'datasets', D.dataset.value_counts().to_dict(), flush=True)
    A = pd.concat(Parallel(n_jobs=2)(delayed(audit)(c, g) for c, g in cells), ignore_index=True)
    A.to_csv('outputs/out_ext_audit.csv', index=False)
    C = A.drop_duplicates('cell')
    print('cells audited', len(C), 'median n', C.n.median(), 'eta_target range', C.eta_target.min(), C.eta_target.max(),
          'cells with deterministic target', int((C.eta_target < 1e-3).sum()))
    tab = A.pivot_table(index='feature', columns='class', values='cell', aggfunc='count', fill_value=0)
    tab = (tab.T / tab.sum(1)).T.round(3); tab.to_csv('outputs/out_ext_class_share.csv'); print(tab)
    by_ds = A.groupby(['dataset', 'class']).size().unstack(fill_value=0); by_ds = (by_ds.T / by_ds.sum(1)).T.round(3)
    by_ds.to_csv('outputs/out_ext_class_by_dataset.csv'); print(by_ds)
    # (b) leave-one-cell-out within each dataset
    from sklearn.ensemble import HistGradientBoostingRegressor
    cls = A.set_index(['cell', 'feature'])['class']
    rows = []
    for ds, dd in D[D.cell.isin([c for c, _ in cells])].groupby('dataset'):
        cl = dd.cell.unique()
        for te in cl:
            tr_cells = [c for c in cl if c != te]
            maj = cls.loc[tr_cells].groupby(level='feature').agg(lambda s: s.value_counts().idxmax())
            sets = {'index only': [], 'index + I': [f for f in F if maj.get(f) == 'I'],
                    'index + T': [f for f in F if maj.get(f) == 'T'], 'index + all non-leaking': [f for f in F if maj.get(f) in ('I', 'T', 'N')]}
            tr = dd[dd.cell.isin(tr_cells)].iloc[::3]; ts = dd[dd.cell == te]
            for nm, fs in sets.items():
                X = ['k'] + fs
                p = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[X], tr.capacity).predict(ts[X])
                y = ts.capacity.values
                rows.append(dict(dataset=ds, cell=te, set=nm, n_feat=len(fs), rmse=float(np.sqrt(np.mean((p - y) ** 2))),
                                 r2=float(1 - np.sum((p - y) ** 2) / np.sum((y - y.mean()) ** 2)), nominal=float(y[:5].mean())))
        print(ds, 'LOCO done', flush=True)
    L = pd.DataFrame(rows); L['rmse_pct'] = 100 * L.rmse / L.nominal; L.to_csv('outputs/out_ext_loco.csv', index=False)
    rng = np.random.default_rng(0); summ = []
    for (ds, s), g in L.groupby(['dataset', 'set']):
        v = g.rmse_pct.values; bs = [np.median(rng.choice(v, len(v))) for _ in range(2000)]
        summ.append(dict(dataset=ds, set=s, cells=len(v), median_rmse_pct=float(np.median(v)), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)),
                         mean_n_feat=float(g.n_feat.mean())))
    S = pd.DataFrame(summ); S.to_csv('outputs/out_ext_loco_summary.csv', index=False); print(S.round(3).to_string(index=False))


In [ ]:
!PYTHONPATH=. python 11_external.py

## Step 6 – index-only ceiling, feature atlas, family transfer (methods Steps 3, 5, 6; Figs. 2, 4, 6)

In [ ]:
%%writefile 17_rineng_analyses.py
"""Step 17 - analyses of the Results in Engineering research article (beyond the MethodsX validation).
(a) Index-attainable accuracy: within-record power-law extrapolation (S2 ceiling) for GMP, the 4 NASA cells and
    374 public cells; GMP remaining-useful-life comparison with the companion LSTM study.
(b) Feature atlas: share of class-I verdicts per charge feature and dataset, by record length and target smoothness.
(c) Transfer value of feature families: within-dataset leave-one-cell-out gradient boosting with
    index only / index + CC-phase / index + CV-phase / index + voltage statistics / index + current statistics.
Outputs: outputs/rineng_*.csv/json and figures/rineng_fig*.png."""
import json, os, sys, warnings, numpy as np, pandas as pd, matplotlib
from scipy import stats
from scipy.optimize import curve_fit, brentq
warnings.filterwarnings('ignore'); sys.path.insert(0, '.')
matplotlib.use('Agg'); import matplotlib.pyplot as plt
QUICK = os.environ.get('TRACE_QUICK') == '1'
O, FG = 'outputs/', 'figures/'
BLUE, ORANGE, GREEN, AMBER, PINK, GREY, INK = '#1f5fbf', '#d9541e', '#138a5e', '#c98a00', '#c2477d', '#8c8c86', '#111111'
DSC = {'XJTU': BLUE, 'TJU': ORANGE, 'HUST': GREEN, 'MIT': PINK}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#dddddd', 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False})
def lab(a, t): a.set_title(t, loc='left', fontsize=15, weight='bold')
def legible(fig, print_w=6.7, min_pt=7.0):
    import matplotlib.text as mt
    need = min_pt * fig.get_figwidth() / print_w
    for t in fig.findobj(mt.Text):
        if t.get_text() and t.get_fontsize() < need: t.set_fontsize(need)

E = pd.read_csv(O + 'out_ext_audit.csv'); A = pd.read_csv(O + 'out_audit_nasa.csv')
FAM = {'CC phase': ['CC Q', 'CC charge time'], 'CV phase': ['CV Q', 'CV charge time'],
       'voltage statistics': ['voltage mean', 'voltage std', 'voltage kurtosis', 'voltage skewness', 'voltage slope', 'voltage entropy'],
       'current statistics': ['current mean', 'current std', 'current kurtosis', 'current skewness', 'current slope', 'current entropy']}
fam_of = {f: k for k, v in FAM.items() for f in v}
R = {}

# ---------------- (a) index-attainable accuracy
d = pd.read_csv('data/GMP.csv'); k = d.CycleNumber.values.astype(float); y = d['Capacity(Ah)'].values; rul = d['RUL(cycles)'].values
pw = lambda k, a, b, c: a - b * k ** c
gmp = []
for h in [500, 1000, 1500]:
    p, _ = curve_fit(pw, k[:h], y[:h], p0=[y[0], 1e-3, 1.2], maxfev=20000)
    eol = brentq(lambda t: pw(t, *p) - 0.7 * y[0], 1, 1e5)
    gmp.append(dict(train_cycles=h, eol=eol, rul_rmse=float(np.sqrt(np.mean((rul[h:] - (eol - k[h:])) ** 2)))))
R['gmp_index_rul'] = gmp; R['cmc_best_lstm_rul_rmse'] = 25.59
C = E.drop_duplicates('cell')[['cell', 'dataset', 'n', 'eta_target', 'R2_index']].copy()
N = A.drop_duplicates('battery')[['battery', 'R2_index', 'eta_target']]
R['nasa_R2_index'] = dict(zip(N.battery, N.R2_index.round(2)))
R['public_R2_index'] = {ds: dict(median=float(g.R2_index.median()), q25=float(g.R2_index.quantile(.25)), q75=float(g.R2_index.quantile(.75)),
                                 share_ge_0_9=float((g.R2_index >= 0.9).mean()), share_lt_0=float((g.R2_index < 0).mean()), cells=len(g))
                        for ds, g in C.groupby('dataset')}
R['public_R2_index_all'] = dict(median=float(C.R2_index.median()), share_ge_0_9=float((C.R2_index >= 0.9).mean()), share_lt_0=float((C.R2_index < 0).mean()))

# ---------------- (b) feature atlas
E['I'] = (E['class'] == 'I').astype(int); E['family'] = E.feature.map(fam_of)
atlas = E.pivot_table(index='feature', columns='dataset', values='I', aggfunc='mean')
atlas['all'] = E.groupby('feature').I.mean(); atlas = atlas.sort_values('all', ascending=False); atlas.round(3).to_csv(O + 'rineng_atlas_feature_dataset.csv')
fam = E.pivot_table(index='family', columns='dataset', values='I', aggfunc='mean'); fam['all'] = E.groupby('family').I.mean(); fam.round(3).to_csv(O + 'rineng_atlas_family.csv')
bins = [0, 300, 600, 1200, 3000]; blab = ['<300', '300–600', '600–1200', '>1200']
E['nbin'] = pd.cut(E.n, bins, labels=blab)
byn = E.groupby(['dataset', 'nbin']).agg(I_share=('I', 'mean'), cells=('cell', 'nunique')).reset_index(); byn.to_csv(O + 'rineng_atlas_length.csv', index=False)
percell = E.groupby('cell').agg(nI=('I', 'sum'), n=('n', 'first'), eta=('eta_target', 'first'), dataset=('dataset', 'first'))
rho_n = stats.spearmanr(percell.n, percell.nI); rho_eta = stats.spearmanr(percell.eta, percell.nI)
R['atlas'] = dict(I_share_all=float(E.I.mean()), cells_with_I=float((percell.nI > 0).mean()),
                  spearman_nI_vs_length=[float(rho_n[0]), float(rho_n[1])], spearman_nI_vs_eta=[float(rho_eta[0]), float(rho_eta[1])],
                  top_features=atlas['all'].head(4).round(3).to_dict(), family_share=fam['all'].round(3).to_dict(),
                  share_by_length={str(b): float(E[E.nbin == b].I.mean()) for b in blab})

# ---------------- (c) transfer value of feature families (leave-one-cell-out within dataset)
from sklearn.ensemble import HistGradientBoostingRegressor
sys.path.insert(0, '.')
import importlib.util
spec = importlib.util.spec_from_file_location('ext', '11_external.py'); src = open('11_external.py').read()
ns = {}; exec(src.split("\nif __name__")[0].split("\nD = load()")[0], ns)   # reuse load() and F without running the audit
FIG_ONLY = os.environ.get('FIG_ONLY') == '1'
D = None if FIG_ONLY else ns['load']()
sets = {'index only': [], **{f'index + {k}': v for k, v in FAM.items()}}
rows = []
for ds, dd in ([] if FIG_ONLY else D[D.cell.isin(C.cell)].groupby('dataset')):
    cl = dd.cell.unique()
    if QUICK: cl = cl[:12]
    for te in cl:
        tr = dd[dd.cell.isin([c for c in cl if c != te])].iloc[::3]; ts = dd[dd.cell == te]; yv = ts.capacity.values
        for nm, fs in sets.items():
            X = ['k'] + fs
            p = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[X], tr.capacity).predict(ts[X])
            rows.append(dict(dataset=ds, cell=te, set=nm, rmse_pct=float(100 * np.sqrt(np.mean((p - yv) ** 2)) / yv[:5].mean())))
    print(ds, 'family LOCO done', flush=True)
L = pd.read_csv(O + 'rineng_family_loco.csv') if FIG_ONLY else pd.DataFrame(rows)
if not FIG_ONLY: L.to_csv(O + 'rineng_family_loco.csv', index=False)
rng = np.random.default_rng(0); summ = []
for (ds, s), g in L.groupby(['dataset', 'set']):
    v = g.rmse_pct.values; bs = [np.median(rng.choice(v, len(v))) for _ in range(2000)]
    summ.append(dict(dataset=ds, set=s, cells=len(v), median=float(np.median(v)), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5))))
S = pd.DataFrame(summ); S.to_csv(O + 'rineng_family_loco_summary.csv', index=False); print(S.round(2).to_string(index=False))
R['family_loco'] = {ds: {r.set: round(r.median, 3) for r in g.itertuples()} for ds, g in S.groupby('dataset')}
json.dump(R, open(O + 'rineng_results.json', 'w'), indent=1, default=float)

# ---------------- figures
# Fig. 2: index-attainable accuracy
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1, 1.25], wspace=0.3))
a = ax[0]; p, _ = curve_fit(pw, k[:500], y[:500], p0=[y[0], 1e-3, 1.2])
a.plot(k, y, color=INK, lw=3, label='GMP capacity (data)'); a.plot(k, pw(k, *p), color=ORANGE, ls='--', lw=2, label='power law, fit on cycles 1–500')
a.axvline(500, color=GREY, ls=':'); a.axhline(0.7 * y[0], color=GREY, ls=':'); a.set_xlabel('cycle'); a.set_ylabel('capacity (Ah)'); a.legend(loc='lower left')
a.text(0.04, 0.47, f"RUL error (cycles)\nindex only: {gmp[0]['rul_rmse']:.2f}\nbest LSTM [CMC]: 25.6", transform=a.transAxes, fontsize=10, va='top', ha='left', bbox=dict(fc='white', ec='0.8', pad=3))
lab(a, '(a)')
a = ax[1]; order = ['XJTU', 'TJU', 'HUST', 'MIT']
for i, ds in enumerate(order):
    v = C[C.dataset == ds].R2_index.clip(-2, 1).values
    a.boxplot(v, positions=[i], widths=0.55, showfliers=False, medianprops=dict(color=INK, lw=2))
    a.scatter(np.full(len(v), i) + np.random.default_rng(i).uniform(-0.18, 0.18, len(v)), v, s=10, color=DSC[ds], alpha=0.6, zorder=3)
a.scatter([4] * 4, N.R2_index.clip(-2, 1), s=40, color=GREY, zorder=3); a.scatter([5], [1.0], s=80, marker='*', color=ORANGE, zorder=3)
a.set_xticks(range(6)); a.set_xticklabels(order + ['NASA', 'GMP']); a.axhline(0, color=INK, lw=0.8)
a.set_ylabel('index-only R² (last 30% of cycles)'); lab(a, '(b)')
legible(fig); fig.savefig(FG + 'rineng_fig2_index_ceiling.png'); plt.close(fig)

# Fig. 3: NASA audit card
feats = ['ambient_T', 't_discharge', 't_Tmax', 'V_600s', 'V_mean', 'T_max', 'T_start', 'I_mean']
names = ['ambient temperature', 'discharge time', 'time to peak temperature', 'voltage after 600 s', 'mean voltage', 'peak temperature', 'start temperature', 'mean current']
CLS = {'D': '#bdbdb5', 'P': '#f5ae8c', 'I': '#a9c6f0', 'T': '#f3d58c', 'N': '#e5e5e0'}
bats = ['B0005', 'B0006', 'B0007', 'B0018']
fig, a = plt.subplots(figsize=(8.5, 5.2)); a.grid(False)
for i, f in enumerate(feats):
    for j, b in enumerate(bats):
        r = A[(A.feature == f) & (A.battery == b)].iloc[0]; c = r['class']
        a.add_patch(plt.Rectangle((j, i), 0.95, 0.9, fc=CLS[c], ec='white'))
        sub = f"|r|={abs(r.r_resid):.3f}" if c == 'P' else (f"S₀={r.S_cond:.2f}" if c in 'IT' else '')
        a.text(j + 0.47, i + 0.33, c, ha='center', va='center', fontsize=14, weight='bold'); a.text(j + 0.47, i + 0.68, sub, ha='center', va='center', fontsize=10)
a.set_xlim(0, 4); a.set_ylim(len(feats), 0); a.set_xticks(np.arange(4) + 0.47); a.set_xticklabels(bats); a.set_yticks(np.arange(len(feats)) + 0.45)
a.set_yticklabels(names); a.tick_params(length=0); [s.set_visible(False) for s in a.spines.values()]
legible(fig); fig.savefig(FG + 'rineng_fig3_nasa_card.png'); plt.close(fig)

# Fig. 4: feature atlas
fig, ax = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw=dict(width_ratios=[1.2, 1], wspace=0.45))
a = ax[0]; a.grid(False); M = atlas[order].values
from matplotlib.colors import LinearSegmentedColormap
cm = LinearSegmentedColormap.from_list('b', ['#ffffff', '#a9c6f0', '#1f5fbf'])
a.imshow(M, cmap=cm, vmin=0, vmax=0.6, aspect='auto')
for i in range(M.shape[0]):
    for j in range(M.shape[1]): a.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=10)
a.set_xticks(range(4)); a.set_xticklabels(order); a.set_yticks(range(len(atlas))); a.set_yticklabels(atlas.index); lab(a, '(a)')
a = ax[1]
for ds in order:
    q = byn[byn.dataset == ds]; q = q[q.cells >= 3]
    a.plot([blab.index(b) for b in q.nbin], q.I_share, marker='o', color=DSC[ds], lw=2.2, label=ds)
a.axhline(0.05, color=GREY, ls='--', lw=1.2); a.set_xticks(range(4)); a.set_xticklabels(blab); a.set_xlabel('record length (cycles)')
a.set_ylabel('share of features classed I'); a.legend(); lab(a, '(b)')
legible(fig); fig.savefig(FG + 'rineng_fig4_atlas.png'); plt.close(fig)

# Fig. 5: transfer value of feature families
fig, a = plt.subplots(figsize=(11, 4.8)); w = 0.16; sl = list(sets)
cols = [GREY, BLUE, GREEN, AMBER, PINK]
for i, s in enumerate(sl):
    q = S[S.set == s].set_index('dataset').reindex(order)
    a.bar(np.arange(4) + (i - 2) * w, q['median'], w, color=cols[i], label=s, yerr=[q['median'] - q.lo, q.hi - q['median']], capsize=2)
a.set_xticks(range(4)); a.set_xticklabels(order); a.set_ylabel('median LOCO RMSE (% of initial capacity)')
a.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
legible(fig); fig.savefig(FG + 'rineng_fig5_family_transfer.png'); plt.close(fig)
print(json.dumps({k: v for k, v in R.items() if k != 'family_loco'}, indent=1, default=float)[:3000])


In [ ]:
!PYTHONPATH=. python 17_rineng_analyses.py

## Step 7 – identifiability vs transfer (methods Step 7; Fig. 7)

In [ ]:
%%writefile 19_rineng_rq4.py
"""Step 19 - RQ4 of the Results in Engineering article: does single-record identifiability (TRACE class-I share of a
feature family in a cell) predict that family's cross-cell transfer value (leave-one-cell-out error reduction)?
Needs outputs of step 11 (out_ext_audit.csv) and step 17 (rineng_family_loco*.csv).
Outputs: outputs/rineng_rq4.json, outputs/rineng_rq4_pairs.csv, figures/rineng_fig6_identifiability_vs_transfer.png."""
import json, numpy as np, pandas as pd, matplotlib, matplotlib.ticker
from scipy import stats
matplotlib.use('Agg'); import matplotlib.pyplot as plt
O, FG = 'outputs/', 'figures/'
DSC = {'XJTU': '#1f5fbf', 'TJU': '#d9541e', 'HUST': '#138a5e', 'MIT': '#c2477d'}
MK = {'CC phase': 'o', 'CV phase': 's', 'voltage statistics': '^', 'current statistics': 'D'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#dddddd', 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False})
FAM = {'CC phase': ['CC Q', 'CC charge time'], 'CV phase': ['CV Q', 'CV charge time'],
       'voltage statistics': ['voltage mean', 'voltage std', 'voltage kurtosis', 'voltage skewness', 'voltage slope', 'voltage entropy'],
       'current statistics': ['current mean', 'current std', 'current kurtosis', 'current skewness', 'current slope', 'current entropy']}
fo = {f: k for k, v in FAM.items() for f in v}
E = pd.read_csv(O + 'out_ext_audit.csv'); L = pd.read_csv(O + 'rineng_family_loco.csv')
S = pd.read_csv(O + 'rineng_family_loco_summary.csv'); A = pd.read_csv(O + 'rineng_atlas_family.csv')
E['fam'] = E.feature.map(fo); E['I'] = (E['class'] == 'I').astype(int)
ci = E.groupby(['cell', 'fam']).agg(Ish=('I', 'mean')).reset_index()
base = L[L.set == 'index only'][['cell', 'rmse_pct']].rename(columns={'rmse_pct': 'b'})
X = L[L.set != 'index only'].copy(); X['fam'] = X.set.str.replace('index + ', '', regex=False)
X = X.merge(base, on='cell').merge(ci, on=['cell', 'fam']); X['log_gain'] = np.log(X.b / X.rmse_pct)
X.to_csv(O + 'rineng_rq4_pairs.csv', index=False)
rng = np.random.default_rng(0)
def rho_ci(g):
    r = stats.spearmanr(g.Ish, g.log_gain); bs = []
    cells = g.cell.unique()
    for _ in range(1000):
        s = rng.choice(cells, len(cells)); gg = pd.concat([g[g.cell == c] for c in s]) if False else g.set_index('cell').loc[s]
        bs.append(stats.spearmanr(gg.Ish, gg.log_gain)[0])
    return float(r[0]), float(r[1]), float(np.nanpercentile(bs, 2.5)), float(np.nanpercentile(bs, 97.5))
R = {'pairs': len(X), 'cells': int(X.cell.nunique()), 'pooled': rho_ci(X), 'per_dataset': {ds: rho_ci(g) for ds, g in X.groupby('dataset')}}
agg = []
for ds in DSC:
    b = S[(S.dataset == ds) & (S.set == 'index only')]['median'].item()
    for f in FAM:
        m = S[(S.dataset == ds) & (S.set == 'index + ' + f)]['median'].item()
        agg.append(dict(dataset=ds, family=f, I_share=A[A.family == f][ds].item(), fold=b / m,
                        improved=float((X[(X.dataset == ds) & (X.fam == f)].log_gain > 0).mean())))
G = pd.DataFrame(agg); r16 = stats.spearmanr(G.I_share, G.fold)
R['dataset_family'] = dict(rho=float(r16[0]), p=float(r16[1]), rows=G.round(3).to_dict('records'))
json.dump(R, open(O + 'rineng_rq4.json', 'w'), indent=1); print(json.dumps({k: R[k] for k in ['pairs', 'pooled', 'per_dataset']}, indent=1)); print(G.round(2))

fig, (a, b) = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw=dict(width_ratios=[1.25, 1], wspace=0.3))
for r in G.itertuples():
    a.scatter(r.I_share, r.fold, s=90, color=DSC[r.dataset], marker=MK[r.family], edgecolor='white', lw=0.8, zorder=3)
a.set_yscale('log'); a.set_ylim(0.8, 9); a.set_yticks([1, 2, 4, 8]); a.set_yticklabels(['1×', '2×', '4×', '8×']); a.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); a.axhline(1, color='#555555', lw=1)
a.set_xlabel('share of family features classed I (single record)'); a.set_ylabel('cross-cell error reduction vs index only')
a.text(0.03, 0.97, f'Spearman ρ = {r16[0]:.2f} (p = {r16[1]:.2f})\n16 dataset–family pairs', transform=a.transAxes, ha='left', va='top', fontsize=11)
h1 = [plt.Line2D([], [], ls='', marker='o', color=c, ms=9, label=d) for d, c in DSC.items()]
h2 = [plt.Line2D([], [], ls='', marker=m, color='#555555', ms=9, label=f) for f, m in MK.items()]
a.legend(handles=h1 + h2, loc='upper right', bbox_to_anchor=(1.0, 0.64), fontsize=10, ncol=2); a.set_title('(a)', loc='left', fontsize=15, weight='bold')
names = list(DSC) + ['pooled']; vals = [R['per_dataset'][d] for d in DSC] + [R['pooled']]
for i, (n, v) in enumerate(zip(names, vals)):
    b.errorbar(i, v[0], yerr=[[v[0] - v[2]], [v[3] - v[0]]], fmt='o', ms=9, color=DSC.get(n, '#111111'), capsize=4, lw=2)
b.axhline(0, color='#555555', lw=1); b.set_xticks(range(len(names))); b.set_xticklabels(names)
b.set_ylabel('Spearman ρ, cell-level I share vs gain'); b.set_ylim(-0.6, 0.6); b.set_title('(b)', loc='left', fontsize=15, weight='bold')
fig.savefig(FG + 'rineng_fig6_identifiability_vs_transfer.png'); print('saved fig6')


In [ ]:
!PYTHONPATH=. python 19_rineng_rq4.py

## Step 8 – robustness: training horizon, equal-size families, random forest (Tables 6, 11)

In [ ]:
%%writefile 20_rineng_robustness.py
"""Step 20 - robustness analyses of the Results in Engineering article.
(a) Index-only ceiling: sensitivity of R2_index to the training fraction (60 / 70 / 80 % of cycles).
(b) Transfer: equal-size control (every family reduced to two features: CC Q + CC time, CV Q + CV time,
    voltage mean + std, current mean + std) and a second learner (random forest) for all input sets.
(c) NASA engineering reading: data for Fig. 3 (restatement of capacity by discharge time; beyond-index co-fluctuation of V600).
Outputs: outputs/rineng_robust_ceiling.csv, rineng_robust_transfer.csv, rineng_robust_transfer_summary.csv, rineng_robust.json."""
import json, os, sys, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore'); sys.path.insert(0, '.')
import trace_audit as T
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from joblib import Parallel, delayed
O = 'outputs/'; QUICK = os.environ.get('TRACE_QUICK') == '1'
ns = {}; exec(open('11_external.py').read().split("\nif __name__")[0].split("\nD = load()")[0], ns)
E = pd.read_csv(O + 'out_ext_audit.csv'); cells = E.drop_duplicates('cell')[['cell', 'dataset']]
D = ns['load'](); D = D[D.cell.isin(cells.cell)]
R = {}

# (a) ceiling vs training fraction
rows = []
for c, g in D.groupby('cell'):
    k = g.k.values.astype(float); y = g.capacity.values
    for f in (0.6, 0.7, 0.8):
        rows.append(dict(cell=c, dataset=g.dataset.iloc[0], n=len(g), train_frac=f, R2_index=T.index_ceiling(k, y, train_frac=f)['R2_index']))
Cc = pd.DataFrame(rows); Cc.to_csv(O + 'rineng_robust_ceiling.csv', index=False)
summ = Cc.groupby(['train_frac', 'dataset']).R2_index.agg(median='median', ge09=lambda v: (v >= 0.9).mean(), lt0=lambda v: (v < 0).mean()).reset_index()
alls = Cc.groupby('train_frac').R2_index.agg(median='median', ge09=lambda v: (v >= 0.9).mean(), lt0=lambda v: (v < 0).mean()).reset_index(); alls['dataset'] = 'All'
R['ceiling_vs_frac'] = pd.concat([summ, alls]).round(3).to_dict('records')
w = Cc.pivot_table(index='cell', columns='train_frac', values='R2_index')
from scipy import stats
R['ceiling_rank_agreement'] = {'rho_60_70': float(stats.spearmanr(w[0.6], w[0.7])[0]), 'rho_70_80': float(stats.spearmanr(w[0.7], w[0.8])[0])}
print(pd.concat([summ, alls]).round(2).to_string(index=False))

# (b) equal-size control and second learner
SETS = {'index only': [], 'index + CC phase': ['CC Q', 'CC charge time'], 'index + CV phase': ['CV Q', 'CV charge time'],
        'index + voltage mean, std': ['voltage mean', 'voltage std'], 'index + current mean, std': ['current mean', 'current std'],
        'index + voltage statistics': ['voltage mean', 'voltage std', 'voltage kurtosis', 'voltage skewness', 'voltage slope', 'voltage entropy'],
        'index + current statistics': ['current mean', 'current std', 'current kurtosis', 'current skewness', 'current slope', 'current entropy']}
def one(ds, dd, te, cl):
    tr = dd[dd.cell.isin([c for c in cl if c != te])].iloc[::3]; ts = dd[dd.cell == te]; yv = ts.capacity.values; out = []
    for nm, fs in SETS.items():
        X = ['k'] + fs
        for lrn, m in (('GBR', HistGradientBoostingRegressor(max_iter=150, random_state=0)),
                       ('RF', RandomForestRegressor(n_estimators=60, min_samples_leaf=5, max_features=0.6, max_samples=0.5, n_jobs=1, random_state=0))):
            if lrn == 'GBR' and nm in ('index only', 'index + CC phase', 'index + CV phase', 'index + voltage statistics', 'index + current statistics'):
                continue   # already in rineng_family_loco.csv (identical settings)
            p = m.fit(tr[X], tr.capacity).predict(ts[X])
            out.append(dict(dataset=ds, cell=te, set=nm, learner=lrn, rmse_pct=float(100 * np.sqrt(np.mean((p - yv) ** 2)) / yv[:5].mean())))
    return out
jobs = []
for ds, dd in D.groupby('dataset'):
    cl = dd.cell.unique()
    if QUICK: cl = cl[:10]
    jobs += [(ds, dd, te, cl) for te in cl]
res = Parallel(n_jobs=-1, verbose=0)(delayed(one)(*j) for j in jobs)
B = pd.DataFrame([r for rr in res for r in rr])
L = pd.read_csv(O + 'rineng_family_loco.csv').assign(learner='GBR')
B = pd.concat([B, L[['dataset', 'cell', 'set', 'learner', 'rmse_pct']]], ignore_index=True); B.to_csv(O + 'rineng_robust_transfer.csv', index=False)
rng = np.random.default_rng(0); S = []
for (ds, s, l), g in B.groupby(['dataset', 'set', 'learner']):
    v = g.rmse_pct.values; bs = [np.median(rng.choice(v, len(v))) for _ in range(1000)]
    S.append(dict(dataset=ds, set=s, learner=l, cells=len(v), median=float(np.median(v)), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5))))
S = pd.DataFrame(S); S.to_csv(O + 'rineng_robust_transfer_summary.csv', index=False)
print(S.pivot_table(index=['learner', 'set'], columns='dataset', values='median').round(2))
# does the RF reproduce the GBR ranking of families? Spearman over the 4 families x 4 datasets of fold gains
fam = ['index + CC phase', 'index + CV phase', 'index + voltage statistics', 'index + current statistics']
P = S.pivot_table(index=['dataset', 'set'], columns='learner', values='median').reset_index()
base = P[P.set == 'index only'].set_index('dataset')
P = P[P.set.isin(fam)].copy(); P['gain_GBR'] = P.apply(lambda r: base.loc[r.dataset, 'GBR'] / r.GBR, axis=1); P['gain_RF'] = P.apply(lambda r: base.loc[r.dataset, 'RF'] / r.RF, axis=1)
R['learner_agreement'] = dict(rho=float(stats.spearmanr(P.gain_GBR, P.gain_RF)[0]), p=float(stats.spearmanr(P.gain_GBR, P.gain_RF)[1]),
                              best_family_RF={ds: g.loc[g.gain_RF.idxmax(), 'set'] for ds, g in P.groupby('dataset')})
# identifiability vs RF gain at the cell level (same test as step 19)
fo = {'CC Q': 'CC phase', 'CC charge time': 'CC phase', 'CV Q': 'CV phase', 'CV charge time': 'CV phase'}
for f in SETS['index + voltage statistics']: fo[f] = 'voltage statistics'
for f in SETS['index + current statistics']: fo[f] = 'current statistics'
E['fam'] = E.feature.map(fo); E['I'] = (E['class'] == 'I').astype(int); ci = E.groupby(['cell', 'fam']).I.mean().rename('Ish').reset_index()
rf = B[B.learner == 'RF']; b0 = rf[rf.set == 'index only'][['cell', 'rmse_pct']].rename(columns={'rmse_pct': 'b'})
X = rf[rf.set.isin(fam)].copy(); X['fam'] = X.set.str.replace('index + ', '', regex=False); X = X.merge(b0, on='cell').merge(ci, on=['cell', 'fam'])
X['lg'] = np.log(X.b / X.rmse_pct); r = stats.spearmanr(X.Ish, X.lg)
R['rf_identifiability_vs_gain'] = dict(rho=float(r[0]), p=float(r[1]), pairs=len(X))
json.dump(R, open(O + 'rineng_robust.json', 'w'), indent=1, default=float); print(json.dumps({k: R[k] for k in ['ceiling_rank_agreement', 'learner_agreement', 'rf_identifiability_vs_gain']}, indent=1))


In [ ]:
!PYTHONPATH=. python 20_rineng_robustness.py

## Step 9 – NASA engineering reading (Fig. 3) and 3-D landscape (Fig. 5) (methods Steps 4, 5)

In [ ]:
%%writefile 21_rineng_extra_figures.py
"""Step 21 - additional figures of the Results in Engineering article.
Fig. 3  NASA engineering reading: (a) capacity vs charge counted from discharge time (restatement, class P);
        (b) cross-fitted residuals of V600 vs residuals of capacity (co-fluctuation beyond the index, class I);
        (c) cross-fitted residuals of mean current vs capacity (instrumentation check).
Fig. 5  3-D landscape of the 374 public cells: record length x index-only ceiling x number of class-I features.
Needs data/nasa_cycle_features.csv and outputs of steps 01, 11."""
import numpy as np, pandas as pd, matplotlib, sys
sys.path.insert(0, '.'); matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy import stats
import trace_audit as T
FG, O = 'figures/', 'outputs/'
DSC = {'XJTU': '#1f5fbf', 'TJU': '#d9541e', 'HUST': '#138a5e', 'MIT': '#c2477d'}
BC = {5: '#1f5fbf', 6: '#d9541e', 7: '#138a5e', 18: '#c2477d'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#dddddd', 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False})
lab = lambda a, t: a.set_title(t, loc='left', fontsize=15, weight='bold')

# ---------------- Fig. 3 NASA engineering reading
N = pd.read_csv('data/nasa_cycle_features.csv')
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw=dict(wspace=0.32))
a = ax[0]
for b, g in N.groupby('battery'):
    q = np.abs(g.I_mean) * g.t_discharge / 3600
    a.scatter(q, g.capacity, s=12, color=BC[b], label=f'B{b:04d}', alpha=0.8)
lim = [1.25, 2.1]; a.plot(lim, lim, color='#111111', lw=1, ls='--'); a.set_xlim(lim); a.set_ylim(lim)
a.set_xlabel('|I| × discharge time (Ah)'); a.set_ylabel('measured capacity (Ah)'); a.legend(loc='upper left', fontsize=10)
r = stats.pearsonr(np.abs(N.I_mean) * N.t_discharge / 3600, N.capacity)[0]
a.text(0.97, 0.05, f'r = {r:.4f}\nclass P in 4/4 cells', transform=a.transAxes, ha='right', fontsize=11); lab(a, '(a)')
rows = []
for b, g in N.groupby('battery'):
    k = g.cycle.values.astype(float)
    ey = T.crossfit_trend(k, g.capacity.values)[1]
    for f in ['V_600s', 'I_mean']:
        ex = T.crossfit_trend(k, g[f].values)[1]
        rows.append(dict(battery=b, feature=f, ey=ey, ex=ex))
for i, (f, t, nm) in enumerate([('V_600s', '(b)', 'V$_{600}$ residual (z)'), ('I_mean', '(c)', 'mean-current residual (z)')]):
    a = ax[i + 1]
    for rr in rows:
        if rr['feature'] != f: continue
        zx = (rr['ex'] - rr['ex'].mean()) / rr['ex'].std(); zy = (rr['ey'] - rr['ey'].mean()) / rr['ey'].std()
        a.scatter(zx, zy, s=12, color=BC[rr['battery']], alpha=0.75)
        rows_rho = stats.spearmanr(zx, zy)[0]; rr['rho'] = rows_rho
    rhos = ', '.join(f"{rr['rho']:.2f}" for rr in rows if rr['feature'] == f)
    a.set_xlabel(nm); a.set_ylabel('capacity residual (z)'); a.set_xlim(-4.5, 4.5); a.set_ylim(-4.5, 4.5)
    a.text(0.03, 0.97, f'Spearman ρ by cell:\n{rhos}', transform=a.transAxes, va='top', fontsize=10.5); lab(a, t)
fig.savefig(FG + 'rineng_fig3_nasa_reading.png'); plt.close(fig)
pd.DataFrame([dict(battery=r['battery'], feature=r['feature'], rho=r['rho']) for r in rows]).to_csv(O + 'rineng_nasa_residual_rho.csv', index=False)
print('saved Fig. 3; residual rho:', [(r['battery'], r['feature'], round(r['rho'], 2)) for r in rows])

# ---------------- Fig. 5 3-D landscape
E = pd.read_csv(O + 'out_ext_audit.csv'); E['I'] = (E['class'] == 'I')
C = E.groupby('cell').agg(n=('n', 'first'), nI=('I', 'sum'), R2=('R2_index', 'first'), ds=('dataset', 'first')).reset_index()
C['R2c'] = C.R2.clip(-1, 1)
fig = plt.figure(figsize=(10.5, 8)); a = fig.add_subplot(111, projection='3d')
for ds in ['XJTU', 'TJU', 'HUST', 'MIT']:
    g = C[C.ds == ds]
    a.scatter(np.log10(g.n), g.R2c, g.nI, s=34, color=DSC[ds], label=f'{ds} ({len(g)} cells)', depthshade=True, edgecolor='white', linewidth=0.3)
# projected shadows on the floor
a.scatter(np.log10(C.n), C.R2c, np.zeros(len(C)) - 0.3, s=6, color='#bbbbbb', alpha=0.5, depthshade=False)
zt = C.nI.max() + 1
xx, yy = np.meshgrid([np.log10(100.0), np.log10(300.0)], [-1, 1]); a.plot_surface(xx, yy, np.full_like(xx, -0.3), color='#d9541e', alpha=0.18, linewidth=0)
a.text2D(0.03, 0.70, 'shaded floor: records < 300 cycles,\nno class-I verdict in any of 79 cells', transform=a.transAxes, fontsize=11, color='#8a2e0b')
a.set_xticks(np.log10([100, 300, 1000, 3000])); a.set_xticklabels(['100', '300', '1000', '3000'])
a.set_xlabel('record length (cycles)', labelpad=10); a.set_ylabel('index-only R² (clipped to ±1)', labelpad=10); a.set_zlabel('class-I features per cell', labelpad=8)
a.set_zlim(-0.3, zt); a.view_init(elev=22, azim=-58); a.legend(loc='upper left', fontsize=11, bbox_to_anchor=(0.02, 0.92))
for p in (a.xaxis.pane, a.yaxis.pane, a.zaxis.pane): p.set_facecolor((0.97, 0.97, 0.97, 1)); p.set_edgecolor('#cccccc')
fig.savefig(FG + 'rineng_fig5_landscape3d.png'); plt.close(fig)
r1 = stats.spearmanr(C.n, C.nI); r2 = stats.spearmanr(C.R2, C.nI); r3 = stats.spearmanr(C.n, C.R2)
print('3D: rho(n,nI)=%.2f rho(R2,nI)=%.2f p=%.3g rho(n,R2)=%.2f p=%.3g' % (r1[0], r2[0], r2[1], r3[0], r3[1]))
pd.Series(dict(rho_n_nI=r1[0], rho_R2_nI=r2[0], p_R2_nI=r2[1], rho_n_R2=r3[0], p_n_R2=r3[1])).to_csv(O + 'rineng_landscape_rho.csv')


In [ ]:
!PYTHONPATH=. python 21_rineng_extra_figures.py

## Step 10 – SHAP explainability and explanation reliability (methods Step 8; Fig. 8)

In [ ]:
%%writefile 22_rineng_explain.py
"""Step 22 - explainability and explanation reliability (Results in Engineering article).
(A) Single-record explanations: for each of the 374 public cells a model is fitted within the record with the common
    random 80/20 row split (index k + 16 charge features -> capacity), and explained with TreeSHAP.
    Measured: random-split R2 (the score such a study would report), share of mean |SHAP| given to the index and to
    features of each TRACE class, whether the top-ranked feature is one the audit can credit (class I),
    and explanation stability (Spearman agreement of the feature ranking between two random splits).
(B) Cross-cell explanations: within each dataset, leave-one-cell-out models on index + 16 features, TreeSHAP on each
    held-out cell; reliability of the feature ranking across folds (Kendall's W) and its agreement with the
    single-record class-I share of each feature.
Outputs: outputs/rineng_xai_within.csv, rineng_xai_cross.csv, rineng_xai.json; figures/rineng_fig8_explainability.png."""
import json, os, sys, warnings, numpy as np, pandas as pd, matplotlib, shap
warnings.filterwarnings('ignore'); sys.path.insert(0, '.'); matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from joblib import Parallel, delayed
O, FG = 'outputs/', 'figures/'; QUICK = os.environ.get('TRACE_QUICK') == '1'
ns = {}; exec(open('11_external.py').read().split("\nif __name__")[0].split("\nD = load()")[0], ns); F = ns['F']
E = pd.read_csv(O + 'out_ext_audit.csv'); D = ns['load'](); D = D[D.cell.isin(E.cell.unique())]
CLS = E.set_index(['cell', 'feature'])['class']
DSC = {'XJTU': '#1f5fbf', 'TJU': '#d9541e', 'HUST': '#138a5e', 'MIT': '#c2477d'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#dddddd', 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False})
X_COLS = ['k'] + F


def within(cell, g):
    out = {}
    imps = []
    for seed in (0, 1):
        rng = np.random.default_rng(seed); idx = rng.permutation(len(g)); cut = int(0.8 * len(g))
        tr, te = g.iloc[idx[:cut]], g.iloc[idx[cut:]]
        m = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[X_COLS], tr.capacity)
        sv = shap.TreeExplainer(m).shap_values(te[X_COLS]); imp = np.abs(sv).mean(0); imps.append(imp)
        if seed == 0: out['R2_random_split'] = r2_score(te.capacity, m.predict(te[X_COLS]))
    imp = imps[0]; tot = imp.sum()
    cls = np.array(['index'] + [CLS[(cell, f)] for f in F])
    for c in ['index', 'I', 'T', 'N', 'P']: out[f'share_{c}'] = float(imp[cls == c].sum() / tot)
    fi = imp[1:]; top = F[int(np.argmax(fi))]
    out.update(cell=cell, dataset=g.dataset.iloc[0], n=len(g), top_feature=top, top_class=CLS[(cell, top)],
               n_I=int((cls[1:] == 'I').sum()), stability=float(stats.spearmanr(imps[0][1:], imps[1][1:])[0]),
               index_rank=int((imp > imp[0]).sum()) + 1)
    return out


cells = list(D.groupby('cell'))
if QUICK: cells = cells[:30]
FIG_ONLY = os.environ.get('FIG_ONLY') == '1'
if FIG_ONLY: W = pd.read_csv(O + 'rineng_xai_within.csv')
else: W = pd.DataFrame(Parallel(n_jobs=-1)(delayed(within)(c, g) for c, g in cells)); W.to_csv(O + 'rineng_xai_within.csv', index=False)


def cross_fold(ds, dd, te, cl):
    tr = dd[dd.cell.isin([c for c in cl if c != te])].iloc[::3]; ts = dd[dd.cell == te]
    if len(ts) > 300: ts = ts.iloc[np.linspace(0, len(ts) - 1, 300).astype(int)]
    m = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[X_COLS], tr.capacity)
    sv = np.abs(shap.TreeExplainer(m).shap_values(ts[X_COLS])).mean(0)
    return dict(dataset=ds, cell=te, **{c: float(v) for c, v in zip(X_COLS, sv)})


jobs = []
for ds, dd in D.groupby('dataset'):
    cl = dd.cell.unique()
    if QUICK: cl = cl[:8]
    jobs += [(ds, dd, te, cl) for te in cl]
if FIG_ONLY: X = pd.read_csv(O + 'rineng_xai_cross.csv')
else: X = pd.DataFrame(Parallel(n_jobs=-1)(delayed(cross_fold)(*j) for j in jobs)); X.to_csv(O + 'rineng_xai_cross.csv', index=False)


def kendall_w(M):  # rows = folds (raters), cols = features
    R = np.apply_along_axis(stats.rankdata, 1, M); m, n = R.shape
    S = ((R.sum(0) - m * (n + 1) / 2) ** 2).sum(); return float(12 * S / (m ** 2 * (n ** 3 - n)))


Ish = E.assign(I=(E['class'] == 'I')).groupby(['dataset', 'feature']).I.mean()
res = {'within': {}, 'cross': {}}
for ds, g in W.groupby('dataset'):
    res['within'][ds] = dict(cells=len(g), R2_random_median=float(g.R2_random_split.median()), share_index=float(g.share_index.median()),
                             share_I=float(g.share_I.mean()), share_T=float(g.share_T.mean()), share_N=float(g.share_N.mean()),
                             top_is_I=float((g.top_class == 'I').mean()), top_is_T=float((g.top_class == 'T').mean()),
                             stability_median=float(g.stability.median()))
res['within']['All'] = dict(cells=len(W), R2_random_median=float(W.R2_random_split.median()), share_index=float(W.share_index.median()),
                            share_I=float(W.share_I.mean()), share_T=float(W.share_T.mean()), share_N=float(W.share_N.mean()),
                            top_is_I=float((W.top_class == 'I').mean()), top_is_T=float((W.top_class == 'T').mean()),
                            stability_median=float(W.stability.median()), R2_ge_0_99=float((W.R2_random_split >= 0.99).mean()),
                            stability_vs_topI=float(stats.mannwhitneyu(W[W.top_class == 'I'].stability, W[W.top_class != 'I'].stability)[1]))
for ds, g in X.groupby('dataset'):
    M = g[F].values; mean_imp = g[F].mean(); rk = mean_imp.sort_values(ascending=False)
    r = stats.spearmanr(mean_imp.values, Ish.loc[ds].reindex(F).values)
    res['cross'][ds] = dict(folds=len(g), kendall_W=kendall_w(M), top3=list(rk.index[:3]), index_share=float((g['k'] / g[X_COLS].sum(1)).median()),
                            rho_importance_vs_Ishare=float(r[0]), p=float(r[1]))
json.dump(res, open(O + 'rineng_xai.json', 'w'), indent=1); print(json.dumps(res, indent=1))

# ---------------- figure
fig, ax = plt.subplots(1, 3, figsize=(17, 5.2), gridspec_kw=dict(width_ratios=[1.15, 1, 1.25], wspace=0.38))
a = ax[0]; order = ['XJTU', 'TJU', 'HUST', 'MIT']; cols = {'index': '#111111', 'I': '#138a5e', 'T': '#8c8c86', 'N': '#c98a00'}
bottom = np.zeros(4)
for c in ['index', 'I', 'T', 'N']:
    v = np.array([W[W.dataset == d][f'share_{c}'].mean() for d in order]); a.bar(order, 100 * v, bottom=100 * bottom, color=cols[c], label=c if c != 'index' else 'cycle index', width=0.6)
    bottom += v
a.set_ylabel('share of mean |SHAP| (%)'); a.set_ylim(0, 118); a.legend(ncol=4, loc='upper center', fontsize=10.5, bbox_to_anchor=(0.5, 1.02), handlelength=1)
a.set_title('(a)', loc='left', fontsize=15, weight='bold')
a = ax[1]
for d in order:
    g = W[W.dataset == d]; a.scatter(g.stability, g.R2_random_split.clip(0.9, 1), s=16, color=DSC[d], alpha=0.75, label=d)
a.set_xlabel('explanation stability between two splits (ρ)'); a.set_ylabel('random-split R² (clipped at 0.9)'); a.legend(fontsize=10, loc='lower left')
a.set_title('(b)', loc='left', fontsize=15, weight='bold')
a = ax[2]
for d in order:
    g = X[X.dataset == d]; mi = g[F].mean(); a.scatter(Ish.loc[d].reindex(F).values, mi.values / mi.max(), s=40, color=DSC[d], label=f"{d}: ρ = {res['cross'][d]['rho_importance_vs_Ishare']:.2f}, W = {res['cross'][d]['kendall_W']:.2f}")
a.set_xlabel('single-record class-I share of feature'); a.set_ylabel('cross-cell mean |SHAP| (relative)'); a.legend(fontsize=9.5, loc='upper right', bbox_to_anchor=(1.0, 0.88))
a.set_title('(c)', loc='left', fontsize=15, weight='bold')
fig.savefig(FG + 'rineng_fig8_explainability.png'); print('saved fig8')


In [ ]:
!PYTHONPATH=. python 22_rineng_explain.py

## Step 11 – methodology figure (Fig. 1)

In [ ]:
%%writefile 18_rineng_fig1.py
"""Step 18 - Fig. 1 of the Results in Engineering article: methodology / study design.
Four zones built from the real data and results: (1) data, (2) audit of each record (TRACE, co-submitted),
(3) analysis steps, (4) verified outcomes and the screening checklist. Run after steps 11, 17 and 19."""
import json, sys, numpy as np, pandas as pd, matplotlib
sys.path.insert(0, '.'); matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import trace_audit as T
O, FG = 'outputs/', 'figures/'
BLUE, ORANGE, GREEN, PINK, GREY, INK, AMBER, PURPLE = '#1f5fbf', '#d9541e', '#138a5e', '#c2477d', '#8c8c86', '#111111', '#c98a00', '#6a3d9a'
DSC = {'XJTU': BLUE, 'TJU': ORANGE, 'HUST': GREEN, 'MIT': PINK}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e6e6', 'savefig.dpi': 300, 'legend.frameon': False})
W, H = 18, 10.2
fig = plt.figure(figsize=(W, H))
bg = fig.add_axes([0, 0, 1, 1]); bg.set_xlim(0, W); bg.set_ylim(0, H); bg.axis('off')


def zone(x, w, col, title):
    bg.add_patch(FancyBboxPatch((x, 0.75), w, 8.25, boxstyle='round,pad=0.02,rounding_size=0.18', fc='white', ec=col, lw=2.5))
    bg.text(x + w / 2, 8.68, title, ha='center', va='center', fontsize=14, weight='bold', color=INK)


def ax_at(x, y, w, h, **kw):
    return fig.add_axes([x / W, y / H, w / W, h / H], **kw)


def tag(a, t, fs=9.5):
    a.set_title(t, loc='left', fontsize=fs, weight='bold')


def arrow(x0, x1, y, txt):
    bg.add_patch(FancyArrowPatch((x0, y), (x1, y), arrowstyle='-|>', mutation_scale=22, lw=2.2, color=GREY))
    bg.text((x0 + x1) / 2, y + 0.18, txt, ha='center', fontsize=9, color='#555555')


# banner
bg.add_patch(FancyBboxPatch((0.2, 9.25), W - 0.4, 0.75, boxstyle='round,pad=0.02,rounding_size=0.15', fc='#fbf1e4', ec=AMBER, lw=1.5))
bg.text(W / 2, 9.62, 'What can one battery ageing record support?  Index ceiling · provenance · identifiable features · cross-cell transfer · explanation reliability',
        ha='center', va='center', fontsize=14, weight='bold')
zx = [(0.2, 3.7, GREEN, '1  Data (Step 1)'), (4.35, 4.15, BLUE, '2  Audit of each record (Step 2)'),
      (8.95, 4.6, PURPLE, '3  Analyses (Steps 3–8)'), (14.0, 3.8, ORANGE, '4  Verified outcomes')]
for z in zx: zone(*z)
arrow(3.92, 4.33, 4.9, ''); arrow(8.52, 8.93, 4.9, ''); arrow(13.57, 13.98, 4.9, '')
bg.text(W / 2, 0.38, 'one row per cycle  →  remove restatements  →  report the index ceiling  →  check record length  →  test transfer on other cells'
        '     |     reproducible Python / Google Colab code', ha='center', fontsize=11, color=PURPLE, weight='bold')

# ---------- zone 1 data
g = pd.read_csv('data/GMP.csv'); a = ax_at(0.75, 6.25, 2.95, 1.75)
a.plot(g.CycleNumber, g['Capacity(Ah)'], color=BLUE, lw=2); a.set_xlabel('cycle', labelpad=1); a.set_ylabel('capacity (Ah)')
tag(a, '(a) GMP LiFePO$_4$ pack, 25.6 V 40 Ah'); a.text(0.04, 0.08, '1 record, 3000 cycles\ngenerator-reproducible', transform=a.transAxes, fontsize=8.5)
N = pd.read_csv('data/nasa_cycle_features.csv'); a = ax_at(0.75, 3.55, 2.95, 1.75)
for b, gg in N.groupby('battery'): a.plot(gg.cycle, gg.capacity, lw=1.3, label=f'B{b:04d}')
a.legend(fontsize=7.5, ncol=2, loc='upper right'); a.set_xlabel('cycle', labelpad=1); a.set_ylabel('capacity (Ah)')
tag(a, '(b) NASA cells: measured, noisy'); a.set_ylim(1.2, 2.25)
ns = {}; exec(open('11_external.py').read().split("\nif __name__")[0].split("\nD = load()")[0], ns)
E = pd.read_csv(O + 'out_ext_audit.csv'); D = ns['load'](); D = D[D.cell.isin(E.cell.unique())]
a = ax_at(0.75, 1.15, 2.95, 1.6); rng = np.random.default_rng(1)
for ds, col in DSC.items():
    cs = D[D.dataset == ds].cell.unique(); pick = rng.choice(cs, min(8, len(cs)), replace=False)
    for c in pick:
        gg = D[D.cell == c]; a.plot(gg.k, gg.capacity / gg.capacity.iloc[:5].mean(), color=col, lw=0.8, alpha=0.8)
a.set_xscale('log'); a.set_xticks([1, 10, 100, 1000]); a.set_xticklabels(['1', '10', '100', '1000']); a.set_xlabel('cycle', labelpad=1); a.set_ylabel('relative capacity'); a.set_ylim(0.6, 1.08)
tag(a, '(c) 374 public cells, 16 charge features')
for i, (ds, col) in enumerate(DSC.items()): a.text(0.03 + 0.24 * i, 0.06, ds, color=col, transform=a.transAxes, fontsize=8.5, weight='bold')

# ---------- zone 2 audit
b5 = N[N.battery == 5]; k = b5.cycle.values.astype(float)
ey = T.crossfit_trend(k, b5.capacity.values)[1]; ev = T.crossfit_trend(k, b5.V_600s.values)[1]
a = ax_at(4.85, 6.25, 3.4, 1.75)
a.plot(k, (ey - ey.mean()) / ey.std(), color=BLUE, lw=1.2, label='capacity residual'); a.plot(k, (ev - ev.mean()) / ev.std(), color=ORANGE, lw=1.0, label='V$_{600}$ residual')
a.legend(fontsize=7.5, loc='upper right'); a.set_xlabel('cycle', labelpad=1); a.set_ylabel('z')
tag(a, '(d) S3: co-fluctuation beyond the index (B0005)')
bg.text(4.6, 5.62, 'S1  provenance: D (index-derived), P (restated target)\nS2  index-only ceiling: power law, R²$_{index}$\n'
        'S3  time-shift test, Holm: I / T / N\nS4  self-surrogate power: MDE$_{80}$', fontsize=9.5, va='top', linespacing=1.5)
sh = E['class'].value_counts(normalize=True).reindex(['T', 'I', 'N', 'P']).fillna(0)
a = ax_at(4.85, 1.05, 3.4, 3.3)
cols = {'T': GREY, 'I': GREEN, 'N': AMBER, 'P': PINK}
a.bar(sh.index, 100 * sh.values, color=[cols[c] for c in sh.index], width=0.6)
for i, v in enumerate(sh.values): a.text(i, 100 * v + 1.5, f'{100 * v:.1f}%', ha='center', fontsize=9)
a.set_ylabel('% of 5984 verdicts'); a.set_ylim(0, 85); tag(a, '(e) audit card classes, public cells')
a.text(0.98, 0.62, 'method and validation:\nco-submitted MethodsX article', transform=a.transAxes, ha='right', fontsize=8.5, style='italic')

# ---------- zone 3 analyses
from scipy.optimize import curve_fit
kk = g.CycleNumber.values.astype(float); yy = g['Capacity(Ah)'].values
p, _ = curve_fit(lambda k, a_, b_, c_: a_ - b_ * k ** c_, kk[:500], yy[:500], p0=[yy[0], 1e-3, 1.2], maxfev=20000)
a = ax_at(9.45, 6.25, 1.75, 1.75); a.plot(kk, yy, color=INK, lw=2.5); a.plot(kk, p[0] - p[1] * kk ** p[2], color=ORANGE, ls='--', lw=1.5)
a.axvline(500, color=GREY, ls=':'); a.set_xticks([0, 1500, 3000]); tag(a, '(f) Step 3: index ceiling'); a.set_xlabel('cycle', labelpad=1)
a.text(0.05, 0.08, 'RUL error\n0.03 cycles', transform=a.transAxes, fontsize=8.5)
fam = pd.read_csv(O + 'rineng_atlas_family.csv').set_index('family').loc[['CC phase', 'CV phase', 'voltage statistics', 'current statistics'], ['XJTU', 'TJU', 'HUST', 'MIT']]
a = ax_at(11.85, 6.25, 1.5, 1.75); a.imshow(fam.values, cmap='Blues', vmin=0, vmax=0.7, aspect='auto'); a.grid(False)
a.set_xticks(range(4)); a.set_xticklabels(['XJ', 'TJ', 'HU', 'MI'], fontsize=8); a.set_yticks(range(4)); a.set_yticklabels(['CC', 'CV', 'V stats', 'I stats'], fontsize=8)
for i in range(4):
    for j in range(4): a.text(j, i, f'{fam.values[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if fam.values[i, j] > 0.4 else INK)
tag(a, '(g) Step 5: atlas', 9.5)
C = E.groupby('cell').agg(n=('n', 'first'), nI=('class', lambda s: (s == 'I').sum()), R2=('R2_index', 'first'), ds=('dataset', 'first'))
a = ax_at(9.05, 2.55, 2.75, 3.1, projection='3d')
for ds, col in DSC.items():
    cc = C[C.ds == ds]; a.scatter(np.log10(cc.n), cc.R2.clip(-1, 1), cc.nI, s=5, color=col, depthshade=True)
a.set_xticks(np.log10([100, 1000])); a.set_xticklabels(['100', '1000'], fontsize=7); a.tick_params(labelsize=7, pad=0)
a.set_xlabel('cycles', fontsize=7.5, labelpad=-6); a.set_ylabel('R²$_{index}$', fontsize=7.5, labelpad=-6); a.set_zlabel('# I', fontsize=7.5, labelpad=-6)
a.set_zticks([0, 10]); a.set_yticks([-1, 0, 1]); a.view_init(elev=20, azim=-58); a.set_title('(h) Step 5: 3-D cell landscape', fontsize=9.5, weight='bold', loc='left')
S = pd.read_csv(O + 'rineng_family_loco_summary.csv'); a = ax_at(12.5, 3.0, 0.95, 2.25)
dss = ['XJTU', 'TJU', 'HUST', 'MIT']
for i, ds in enumerate(dss):
    v0 = S[(S.dataset == ds) & (S.set == 'index only')]['median'].item(); v1 = S[(S.dataset == ds) & (S.set == 'index + voltage statistics')]['median'].item()
    a.bar(i - 0.2, v0, 0.4, color=GREY); a.bar(i + 0.2, v1, 0.4, color=AMBER)
a.set_ylim(0, 4.8); a.set_xticks(range(4)); a.set_xticklabels(['XJ', 'TJ', 'HU', 'MI'], fontsize=8); a.set_ylabel('LOCO RMSE (%)', fontsize=8, labelpad=1)
tag(a, '(i) Step 6: transfer', 9.5); a.text(0.98, 0.92, 'index only', color=GREY, transform=a.transAxes, ha='right', fontsize=7.5, weight='bold')
a.text(0.98, 0.80, '+ voltage stats', color=AMBER, transform=a.transAxes, ha='right', fontsize=7.5, weight='bold')
bg.text(9.3, 2.2, 'Step 4: NASA verdicts → measurement mechanism\nStep 6: leave-one-cell-out GBR / RF, 4 families\n'
        'Step 7: identifiability vs transfer (1496 pairs)\nStep 8: SHAP explanations and their reliability', fontsize=9.3, va='top', linespacing=1.5)

# ---------- zone 4 outcomes
rq4 = json.load(open(O + 'rineng_rq4.json')); R = json.load(open(O + 'rineng_results.json')); X = json.load(open(O + 'rineng_xai.json'))['within']['All']
XAI_TXT = f"stable SHAP (ρ = {X['stability_median']:.2f}), yet top feature\nnot creditable in {100 * (1 - X['top_is_I']):.0f}% of records"
items = [(ORANGE, 'Index ceiling', f"GMP RUL error 0.03 cycles (LSTM 25.6)\nR²$_{{index}}$ ≥ 0.9 in {100 * R['public_R2_index_all']['share_ge_0_9']:.1f}% of cells"),
         (PINK, 'Restatements', 't$_{dis}$, t$_{Tmax}$ = class P in 4/4 NASA cells\nV$_{600}$ = class I in 4/4 cells'),
         (GREEN, 'Identifiable features', f"{100 * R['atlas']['I_share_all']:.1f}% of verdicts; CV phase highest\nnone in 79 records < 300 cycles"),
         (BLUE, 'Identifiability ≠ transfer', f"ρ = {rq4['pooled'][0]:.2f} (1496 pairs)\nvoltage stats: up to 6.2-fold gain"),
         (PURPLE, 'Explanations ≠ evidence', XAI_TXT)]
for i, (col, t, b) in enumerate(items):
    y0 = 7.5 - i * 1.13
    bg.add_patch(FancyBboxPatch((14.25, y0), 3.3, 0.98, boxstyle='round,pad=0.02,rounding_size=0.1', fc='white', ec=col, lw=1.8))
    bg.text(14.4, y0 + 0.76, t, fontsize=10.5, weight='bold', color=col, va='center')
    bg.text(14.4, y0 + 0.52, b, fontsize=8.5, va='top', linespacing=1.35)
bg.add_patch(FancyBboxPatch((14.25, 0.95), 3.3, 1.85, boxstyle='round,pad=0.02,rounding_size=0.1', fc='#f3f0f8', ec=PURPLE, lw=1.8))
bg.text(14.4, 2.6, 'Screening checklist (Section 5)', fontsize=10.5, weight='bold', color=PURPLE, va='center')
bg.text(14.4, 2.38, '1  remove restated columns\n2  report R²$_{index}$ beside every score\n3  require a measured target\n'
        '4  ≥ 300 cycles; report MDE$_{80}$\n5  test transfer on other cells', fontsize=9, va='top', linespacing=1.4)
fig.savefig(FG + 'rineng_fig1_study_design.png'); print('saved figures/rineng_fig1_study_design.png')


In [ ]:
!PYTHONPATH=. python 18_rineng_fig1.py

## Step 12 – view figures and key numbers

In [ ]:
from IPython.display import Image, display
import glob, json
for f in sorted(glob.glob('figures/rineng_*.png')): print(f); display(Image(f, width=900))
for j in ['rineng_results.json', 'rineng_rq4.json', 'rineng_robust.json', 'rineng_xai.json']:
    print(j); print(json.dumps(json.load(open('outputs/' + j)), indent=1)[:3000])

In [ ]:
!zip -qr TRACE_RINENG_outputs.zip outputs figures && echo 'download TRACE_RINENG_outputs.zip from the Files pane'